# Dwelling Classifier Pipeline
## New workflow (feature_store + FeatureSetConfig)

**Step 1** — Run `build_full_cache()` ONCE (or when raw data changes).  
**Step 2** — Run `update_cache()` ONLY when you add new features to the logic file.  
**Step 3** — Declare a `FeatureSetConfig` for each experiment.  
**Step 4** — `train → infer → predict → assess` using that config.  

The cache is **annotation-free and split-ignorant** — it stores every feature
for every frame of every source.  Labels and train/val splits are applied
inside `train()` and `infer()` by joining `ctx.annotated` at run-time.

In [1]:
# ── CELL 0  Imports & paths ───────────────────────────────────────────────────
import importlib
import random
import os
import gc
import shutil
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

import feature_store as fs
importlib.reload(fs)
import tp_export as tp
from tp_export import PostProcessConfig
import split_dataset as sd
from pp import get_context, validate_context

# ── Paths (edit to match) ─────────────────────────────────
TRAIN_DATA   = Path(r"C:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\annotations_split\train")
ANN_MASTER   = Path(r"C:\Users\corna\honours\fresh1\hp_2\data_intermediate\annotation\annotation.csv")
BASE_OUT_DIR = Path(r"C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models")

# Master feature cache — shared across ALL experiments
# One parquet per source, no annotation data, never needs rebuilding unless
# raw trajectory data changes or you add new base features.
MASTER_CACHE_DIR = BASE_OUT_DIR / "master_feature_cache"

LOGIC_FILE = "feature_registry"
FPS        = 6.0

SESSION_DIR = TRAIN_DATA / "train"
VAL_DIR     = TRAIN_DATA / "validation"

In [ ]:
# ── CELL 1  Split dataset ────────────────────────────────────
seed = random.randint(0, 4294967295)
print(seed)
importlib.reload(sd)
sd.split_dataset(TRAIN_DATA, TRAIN_DATA, mode="train", RANDOM_SEED=seed)

In [2]:
# ── CELL 2  Build context ────────────────────────────────────
train_ann = pd.read_csv(SESSION_DIR / "annotations_train.csv")
val_ann   = pd.read_csv(VAL_DIR     / "annotations_validation.csv")

TRAIN_PREFIXES = train_ann["source"].unique().tolist()
TEST_FILES     = val_ann["source"].unique().tolist()
sources        = list(set(TRAIN_PREFIXES).union(set(TEST_FILES)))

TRAIN_SLICES = {src: slice(None) for src in TRAIN_PREFIXES}
train_keys   = train_ann[["source", "ID"]].drop_duplicates()
val_keys     = val_ann[["source", "ID"]].drop_duplicates()
all_ann_keys = pd.merge(train_keys,val_keys,how="outer")

ctx = get_context(
    parquet_path = Path(r"C:\Users\corna\honours\fresh1\hp_2\data_intermediate\raw_trajectories.parquet"),
    ann_csv      = ANN_MASTER,
    session_dir  = TRAIN_DATA,
    sources      = sources,
)
clean_full_annotations = ctx.annotated.copy()


Loading raw trajectories from parquet...
  → 16,859,236 framessources: ['EA', 'GA1', 'GA2', 'GA3', 'H2O']
Loading annotations...
  → 891 annotation intervals
      nondwelling: 513
      dwelling: 338
      turn: 20
      crawling: 20
Labelling frames...
  → 222,467 labeled frames
      nondwelling: 131,805 frames
      dwelling: 87,913 frames
      crawling: 2,178 frames
      turn: 571 frames


In [3]:
#set single PPC and single HP set

# Post-processing config
MF, TH, GC, GO = 15, 0.61, 4, 14

#Hyperparameters
max_feat = 0.3
max_dep = 35
n_est = 150
min_samples = 0.0015987
min_split = (0.0015987 * 3.9163778)

ppc    = PostProcessConfig(MF, TH, GC, GO)
ppc_id = ppc.get_IDstr()

print(f"Train sources : {TRAIN_PREFIXES}")
print(f"Val sources   : {TEST_FILES}")

@dataclass
class Hyperparameters:
    name: str = "unnamed"
    max_features: float = 0.3
    min_samples_leaf: float = 0.0015987
    min_samples_split: float = (0.0015987 * 3.9163778)
    num_estimators: float = 150
    max_depth: float = 35
    
    def get_id(self):
        return f"NE{self.num_estimators}_ML{self.min_samples_leaf}_MS{self.min_samples_split}_MF{self.max_features}_MD{self.max_depth}"
    
hp = Hyperparameters(max_feat,min_samples,min_split,n_est,max_dep)

Train sources : ['GA1', 'GA3', 'GA2', 'H2O', 'EA']
Val sources   : ['GA1', 'GA3', 'GA2', 'H2O', 'EA']


---
## Step 1 — Build master feature cache (run once)

Computes **every feature** for **every frame** of **every source**.  
No annotations, no labels, no split awareness.  
Pass `force=True` to rebuild (e.g. after raw parquet changes).  
Pass `sources=[...]` to add only new sources to an existing cache.

In [ ]:
# ── CELL 3  Build full cache (ONCE) ──────────────────────────────────────────
importlib.reload(fs)

fs.build_full_cache(
    ctx        = ctx,
    logic_file = LOGIC_FILE,
    cache_dir  = MASTER_CACHE_DIR,
    fps        = FPS,
    windows = [10],
    # sources  = ["H2O"],   # uncomment to add a single new source
    # force    = True,       # uncomment to force-recompute all sources
)

# Health check
fs.check_cache_health(MASTER_CACHE_DIR, ctx=ctx, logic_file=LOGIC_FILE)

---
## Step 2 — Update cache (optional — only when logic file changes)

If you add a new feature to `exp_feature_calculation.py`.  
It appends only the new columns — never rewrites existing ones.

In [ ]:
# ── CELL 4  Update cache with new features (OPTIONAL) ─────────────────────────
importlib.reload(fs)

# Option A — let the system auto-detect missing columns:
# fs.update_cache(ctx, LOGIC_FILE, MASTER_CACHE_DIR, fps=FPS)

# Option B — explicitly name the new columns you added:
# fs.update_cache(ctx, LOGIC_FILE, MASTER_CACHE_DIR, fps=FPS,
#                 new_columns=["w30_my_new_feature", "w75_my_new_feature"])

# Option C — bump version of a column whose formula changed:
# fs.update_cache(ctx, LOGIC_FILE, MASTER_CACHE_DIR, fps=FPS,
#                 version_bump={"w50_tortuosity": 2})

# Browse what's in the cache:
cached_features = fs.list_cached_features(MASTER_CACHE_DIR)
print(cached_features.to_string())

---
## Step 3 — Declare a FeatureSetConfig

This is the **only thing you change** between experiments.  
The config is saved alongside the model so you always know what went in.

In [31]:
# ── CELL 5  Define experiment feature set ─────────────────────────────────────

importlib.reload(fs)
from feature_store import FeatureSetConfig
                     
fsc = FeatureSetConfig(
    name="optima0623",
    description="optima picks",
    base_features=[],
    #base_features=["larva_body_length"],
    windowed_features={
        # present at every window
        #"omega_body_mean":     [10],
        #"omega_head_std":      [95,110,115,120],
        "omega_relative_mean": [5],
        "rog":                 [43],
        "tortuosity":          [95],
        #"msd":                 [70,110,120],
        "bending_std":         [50],
        #"hc_ratio_mean":       [10,50,55,120],
        #"ht_ratio_mean":       [120],
        #"omega_body_mean_slope_smooth": [65],
        "vel_mean":            [5],
        #"vel_std":             [25,40,115],
        "vel_norm_mean":       [43,],
        "head_vel_mean":       [30],
        #"head_vel_std":        [30, 50, 75],

        #"omega_heading_mean":  [30, 50, 75],
        #"coverage":            [15,40], 
        "bend_peaks_rate":     [23,95],
        #"pause_run_frac":      [80],
        #"reversal_rate":       [40,110,115],
        "angular_tortuosity":  [43,65],
        "revisitation_mean":   [23],
        #"revis_slope_smooth":  [43],
        "rog_slope_smooth":    [5,50,95],
        #"tort_slope_smooth":   [55,105,120],
        #"vel_lag":             [50],
        #"vel_lead":            [115],
        "bend_freq_rolling":   [75],
        "path_efficiency": [50],
        #"vel_autocorr": [65,70],
        #"pca_ratio": [50],
        #"pca_area_norm": [30],
        #"net_displacement_norm": [17],
        "turn_fraction": [50],
        "posture_asymmetry_max": [5],
        #"pause_fraction": standard_wins,
        #"omega_heading_max": standard_wins,
        #"curvature_index_mean": [80],
        #"head_bend_max": [10],
        "path_length_norm": [95],
        "turn_event_alternation": [23,43],
        #"turn_event_interval_cv": heavy_wins,
        #"turn_event_amp_mean": heavy_wins,
        "turn_event_gap_max_bl": [23],
        #"turn_event_gap_mean_bl": heavy_wins,
        #"turn_event_count": heavy_wins,
    },
    
)
# Example: full feature set (same as the original hard-coded set)
# fsc = FeatureSetConfig(
#     name="optima 06-23",
#     description="optima picks",
#     base_features=[],
#     #base_features=["larva_body_length"],
#     windowed_features={
#         # present at every window
#         "omega_body_mean":     [10],
#         #"omega_head_std":      [95,110,115,120],
#         #"omega_relative_mean": [5],
#         "rog":                 [43,95,120],
#         "tortuosity":          [95],
#         "msd":                 [70,110,120],
#         "bending_std":         [50, 80, 110],
#         #"hc_ratio_mean":       [10,50,55,120],
#         #"ht_ratio_mean":       [120],
#         "omega_body_mean_slope_smooth": [65],
#         "vel_mean":            [23,50],
#         #"vel_std":             [25,40,115],
#         "vel_norm_mean":       [43,80,110],
#         "head_vel_mean":       [30,50,65],
#         #"head_vel_std":        [30, 50, 75],
#         #"omega_heading_mean":  [30, 50, 75],
#         #"coverage":            [15,40],
#         "bend_peaks_rate":     [23,95],
#         "pause_run_frac":      [65,80],
#         #"reversal_rate":       [40,110,115],
#         #"angular_tortuosity":  [43],
#         #"revisitation_mean":   [23],
#         "revis_slope_smooth":  [43],
#         "rog_slope_smooth":    [50,95],
#         #"tort_slope_smooth":   [55,105,120],
#         #"vel_lag":             [50],
#         "vel_lead":            [110,115,120],
#         "bend_freq_rolling":   [75],
#         "path_efficiency": [13,50],
#         "vel_autocorr": [43,65,70],
#         "pca_ratio": [43,50],
#         "pca_area_norm": [30],
#         "net_displacement_norm": [17,43,50],
#         "turn_fraction": [110],
#         "posture_asymmetry_max": [5,10,13],
#         #"pause_fraction": standard_wins,
#         #"omega_heading_max": standard_wins,
#         "curvature_index_mean": [80],
#         "head_bend_max": [10,13],
#         "path_length_norm": [95,110,120],
#         "turn_event_alternation": [43],
#         #"turn_event_interval_cv": heavy_wins,
#         #"turn_event_amp_mean": heavy_wins,
#         "turn_event_gap_max_bl": [23],
#         #"turn_event_gap_mean_bl": heavy_wins,
#         #"turn_event_count": heavy_wins,
#     },
# )

# Example: ablation — velocity features only
# fsc = FeatureSetConfig(
#     name="velocity_only",
#     description="Ablation: only velocity-based features across all windows",
#     base_features=["larva_body_length"],
#     windowed_features={
#         "vel_mean":       [30, 50, 75],
#         "vel_std":        [30, 50, 75],
#         "head_vel_mean":  [30, 50, 75],
#         "head_vel_std":   [30, 50, 75],
#         "vel_lag":        [50, 75],
#         "vel_lead":       [50, 75],
#     },
# )

print(fsc.summary())

# Validate against the cache — auto-fills any missing columns
fsc.validate_against_cache(
    MASTER_CACHE_DIR,
    ctx=ctx,
    logic_file=LOGIC_FILE,
    fps=FPS,
    auto_fill=True,
)

FeatureSetConfig 'optima0623'
  Description   : optima picks
  Base features : []
  Windowed      :
    omega_relative_mean            windows=[5]
    rog                            windows=[43]
    tortuosity                     windows=[95]
    bending_std                    windows=[50]
    vel_mean                       windows=[5]
    vel_norm_mean                  windows=[43]
    head_vel_mean                  windows=[30]
    bend_peaks_rate                windows=[23, 95]
    angular_tortuosity             windows=[43, 65]
    revisitation_mean              windows=[23]
    rog_slope_smooth               windows=[5, 50, 95]
    bend_freq_rolling              windows=[75]
    path_efficiency                windows=[50]
    turn_fraction                  windows=[50]
    posture_asymmetry_max          windows=[5]
    path_length_norm               windows=[95]
    turn_event_alternation         windows=[23, 43]
    turn_event_gap_max_bl          windows=[23]
  Total columns : 23

[]

---
## Step 4 — Train → Infer → Predict → Assess

In [32]:
# ── CELL 6  Output paths ──────────────────────────────────────────────────────
importlib.reload(tp)
#SEED = seed
try:
    SEED = seed
except NameError:
    SEED = random.randint(0, 4294967295)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")#"20260623_142258" #datetime.now().strftime("%Y%m%d_%H%M%S") #"20260623_111754"# datetime.now().strftime("%Y%m%d_%H%M%S") #"20260623_111754" #datetime.now().strftime("%Y%m%d_%H%M%S") #"20260622_104847" # datetime.now().strftime("%Y%m%d_%H%M%S")
prefixes = list(TRAIN_SLICES.keys())
train_str = "-".join(prefixes)
test_str = "-".join(prefixes)

model_folder_name = (
    f"RUN_{LOGIC_FILE}_{fsc.name}_{RUN_ID}_train-{train_str}_test-{test_str}"
)
final_output_path = BASE_OUT_DIR / model_folder_name

model_path_ext         = "final_model.pkl"
probabilities_path_ext   = "probabilities.csv"
predictions_dir_ext      = "predictions"
feature_path_ext         = "feature_set_config.pkl"   # stores FSC object
metadata_path_ext        = "metadata.pkl"
metadata_test_path_ext   = "metadata_test.pkl"
plot_path_ext            = "plots"
report_path_ext          = "experiment_log.txt"

print(f"Outputs → {final_output_path}")

Outputs → C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models\RUN_feature_registry_optima0623_20260623_172558_train-GA1-GA3-GA2-H2O-EA_test-GA1-GA3-GA2-H2O-EA


In [62]:
# OPTIONAL : TRAIN/INFER/PREDICT/ASSESS multiple ppcs or hyperparameters or featuresets

ppcs = []

#ppcs.append(PostProcessConfig(MF,0.575,GC,GO))
ppcs.append(PostProcessConfig(MF,0.6,GC,GO))
#ppcs.append(PostProcessConfig(MF,0.625,GC,GO))


fscs = []

standard_wins = [5,10,13,17,23,30,43,50,65,70,80,95,110,120]
standard_features = ["omega_body_mean","omega_head_std","omega_relative_mean","rog","tortuosity","msd","bending_std","hc_ratio_mean","ht_ratio_mean","omega_body_mean_slope_smooth","vel_mean","vel_std","vel_norm_mean","head_vel_mean",
                     "head_vel_std", "omega_heading_mean","coverage","bend_peaks_rate","pause_run_frac","reversal_rate","angular_tortuosity","revisitation_mean","revis_slope_smooth",
                     "rog_slope_smooth","tort_slope_smooth","vel_lag","vel_lead","bend_freq_rolling","path_efficiency","vel_autocorr","pca_ratio","pca_area_norm","net_displacement_norm","turn_fraction",
                     "posture_asymmetry_max","pause_fraction","omega_heading_max","curvature_index_mean","head_bend_max","path_length_norm","turn_event_alternation","turn_event_interval_cv",
                     "turn_event_amp_mean","turn_event_gap_max_bl","turn_event_gap_mean_bl","turn_event_count"]
                     
features = FeatureSetConfig(name="window_30_full")
FeatureSetConfig.windowed_features = {}
for feat in standard_features:
    features.windowed_features.update({feat:[30]})
    
fscs.append(features)


hps = [hp]


In [63]:
# ── CELL 7  Train ─────────────────────────────────────────────────────────────
importlib.reload(tp)
#importlib.reload(fs)
#from feature_store import FeatureSetConfig

print("\n--- INITIATING TRAINING ---")
ctx.annotated = clean_full_annotations.copy()

for feature_set in fscs:
    curr_path = final_output_path / feature_set.get_id()
    
    for hyperparams in hps:
        curr_path = final_output_path / feature_set.get_id() / hyperparams.get_id()
        
        os.makedirs(curr_path, exist_ok=True)
        
        report_path = curr_path / report_path_ext
        
        predictions_dir = curr_path / predictions_dir_ext
        os.makedirs(predictions_dir, exist_ok=True)

        
        with open(report_path, "a") as f:
            f.write(f"Session initialized at: {datetime.now()}\n")
            f.write(f"Logic file: {LOGIC_FILE}\n")
            f.write(f"Feature set: {feature_set.name}\n")
            f.write(f"Hyperparameters: {hyperparams.name}")
            f.write(feature_set.summary() + "\n\n")
            
        os.makedirs(predictions_dir, exist_ok=True)
        os.makedirs(curr_path, exist_ok=True)
        
        model, features, metadata, log_messages = tp.train(
            ctx          = ctx,
            slices       = TRAIN_SLICES,
            prefixes     = prefixes,
            logic_file   = LOGIC_FILE,
            model_path   = curr_path / model_path_ext,
            feature_path = curr_path/ feature_path_ext,
            metadata_path= curr_path/ metadata_path_ext,
            seed         = SEED,
            cache_dir    = MASTER_CACHE_DIR,
            fsc          = feature_set,
            train_keys   = train_keys,      # ← train() pre-filters annotations to these larvae
            do_plot_gini    = True,             # plot feature importances,
            do_plot_shap=True,shap_subsample=10000,
            do_spearman_clustering=True,spearman_corr_threshold=0.85,enact_clustering=False,
            plot_path    = curr_path / plot_path_ext,
            max_feat = hyperparams.max_features,
            max_dep = hyperparams.max_depth,
            n_est = hyperparams.num_estimators,
            min_samples = hyperparams.min_samples_leaf,
            min_split = hyperparams.min_samples_split,
        )


--- INITIATING TRAINING ---
[FeatureSetConfig 'window_30_full'] ⚠ 3 columns missing from cache: ['w30_vel_lag', 'w30_vel_lead', 'w30_bend_freq_rolling']
  Auto-filling missing columns via update_cache() …
[feature_store] Updating cache for 5 sources …
  [EA] targeted compute for 3 column(s)
  [calculate_columns] Window 30 — computing: ['bend_freq_rolling', 'vel_lag', 'vel_lead']
    SLOW: w30_bend_freq_rolling took 8.7s
  [EA] saved 3 columns to fragment batch_1782252814
  [GA1] targeted compute for 3 column(s)
  [calculate_columns] Window 30 — computing: ['bend_freq_rolling', 'vel_lag', 'vel_lead']
    SLOW: w30_bend_freq_rolling took 11.2s
  [GA1] saved 3 columns to fragment batch_1782252814
  [GA2] targeted compute for 3 column(s)
  [calculate_columns] Window 30 — computing: ['bend_freq_rolling', 'vel_lag', 'vel_lead']
    SLOW: w30_bend_freq_rolling took 13.7s
  [GA2] saved 3 columns to fragment batch_1782252814
  [GA3] targeted compute for 3 column(s)
  [calculate_columns] Window

In [64]:
# ── CELL 8  Infer ─────────────────────────────────────────────────────────────
importlib.reload(tp)

#ctx.annotated = clean_full_annotations.merge(val_keys, on=["source", "ID"], how="inner")

for feature_set in fscs:
    curr_path = final_output_path / feature_set.get_id()
    
    for hyperparams in hps:
        curr_path = final_output_path / feature_set.get_id() / hyperparams.get_id()
        report_path = curr_path / report_path_ext
        predictions_dir = curr_path / predictions_dir_ext
        
        with open(report_path, "a") as f:
            f.write(f"Inferred at: {datetime.now()}\n")
            
        os.makedirs(predictions_dir, exist_ok=True)
        os.makedirs(curr_path, exist_ok=True)
        
        metadata_test, log_messages_inf = tp.infer(
            model_path         = curr_path / model_path_ext,
            ctx                = ctx,
            files              = TEST_FILES,
            feature_path       = curr_path / feature_path_ext,
            probabilities_path = curr_path / probabilities_path_ext,
            metadata_test_path = curr_path / metadata_test_path_ext,
            cache_dir          = MASTER_CACHE_DIR,
            logic_file         = LOGIC_FILE,
            #fsc                = feature_set,             
            )

Model loaded from C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models\RUN_feature_registry_optima0623_20260623_172558_train-GA1-GA3-GA2-H2O-EA_test-GA1-GA3-GA2-H2O-EA\total46_unique46_1111111111111111111111111111111111111111111111_-8658066907567470354\NE35_ML0.006261113188859999_MS150_MF0.0015987_MD35\final_model.pkl

[infer] FeatureSetConfig 'window_30_full': 46 features
[FeatureSetConfig 'window_30_full'] ✓ All 46 columns present in cache.

=== Processing file_id=GA1 ===
  Source: GA1  |  Rows: 3,518,524
  RAM free: 5.5 GB
  Matched 33411 annotated frames.
  Probabilities appended → C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models\RUN_feature_registry_optima0623_20260623_172558_train-GA1-GA3-GA2-H2O-EA_test-GA1-GA3-GA2-H2O-EA\total46_unique46_1111111111111111111111111111111111111111111111_-8658066907567470354\NE35_ML0.006261113188859999_MS150_MF0.0015987_MD35\probabilities.csv
  RAM free after cleanup: 7.0 GB

=== Processing file_id=GA3 ===
  Sour

In [65]:
# OPTIONAL: PERMUTATION IMPORTANCE
importlib.reload(tp)


for feature_set in fscs:
    curr_path = final_output_path / feature_set.get_id()
    
    for hyperparams in hps:
        curr_path = final_output_path / feature_set.get_id() / hyperparams.get_id()
        report_path = curr_path / report_path_ext
        predictions_dir = curr_path / predictions_dir_ext
        plot_path = curr_path / plot_path_ext
        
        with open(report_path, "a") as f:
            f.write(f"Permutated at: {datetime.now()}\n")
            
        os.makedirs(predictions_dir, exist_ok=True)
        os.makedirs(curr_path, exist_ok=True)

        df_all_weights, df_noise = tp.evaluate_validation_permutation_importance(
            model_path=curr_path / model_path_ext,
            ctx=ctx,
            val_prefixes=TEST_FILES,
            logic_file=LOGIC_FILE,
            feature_path=curr_path / feature_path_ext,
            cache_dir=MASTER_CACHE_DIR,
            val_keys=val_keys,
            report_path=report_path,
            plots_dir=plot_path,
            n_repeats=10
        )

Model loaded from C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models\RUN_feature_registry_optima0623_20260623_172558_train-GA1-GA3-GA2-H2O-EA_test-GA1-GA3-GA2-H2O-EA\total46_unique46_1111111111111111111111111111111111111111111111_-8658066907567470354\NE35_ML0.006261113188859999_MS150_MF0.0015987_MD35\final_model.pkl

[Permutation Importance] Reconstructing validation set for 46 features...
[Permutation Importance] Shuffling 45,732 validation rows (10 passes per feature)...
[Permutation Importance] Saved plot   → C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models\RUN_feature_registry_optima0623_20260623_172558_train-GA1-GA3-GA2-H2O-EA_test-GA1-GA3-GA2-H2O-EA\total46_unique46_1111111111111111111111111111111111111111111111_-8658066907567470354\NE35_ML0.006261113188859999_MS150_MF0.0015987_MD35\plots\validation_permutation_importance.png
[Permutation Importance] Logged to    → C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models\RUN_featu

In [29]:
# ── CELL 9  Predict (post-process) ────────────────────────────────────────────
importlib.reload(tp)

for feature_set in fscs:
    curr_path = final_output_path / feature_set.get_id()
    
    for hyperparams in hps:
        curr_path = final_output_path / feature_set.get_id() / hyperparams.get_id()
        report_path = curr_path / report_path_ext
        predictions_dir = curr_path / predictions_dir_ext
        probabilities_path = curr_path / probabilities_path_ext
        
        with open(report_path, "a") as f:
            f.write(f"Predicted at: {datetime.now()}\n")
            
        os.makedirs(predictions_dir, exist_ok=True)
        os.makedirs(curr_path, exist_ok=True)
        
        for pcon in ppcs:

            predictions = tp.predict(
                probabilities_path = probabilities_path,
                ppc                = pcon,
                predictions_dir    = predictions_dir,
                ctx                = ctx,
                logic_file         = LOGIC_FILE,
        plot               = False,
    )

c:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\tp_export.py:618: DtypeWarning: Columns (0: behavior, 1: tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df_probs = pd.read_csv(probabilities_path)


Predicting for EA track 0…
Predicting for EA track 1000…
Predicting for EA track 2000…
Predicting for EA track 3000…
Predicting for GA1 track 0…
Predicting for GA1 track 1000…
Predicting for GA1 track 2000…
Predicting for GA1 track 3000…
Predicting for GA1 track 4000…
Predicting for GA1 track 5000…
Predicting for GA1 track 6000…
Predicting for GA2 track 0…
Predicting for GA2 track 1000…
Predicting for GA2 track 2000…
Predicting for GA2 track 3000…
Predicting for GA2 track 4000…
Predicting for GA2 track 5000…
Predicting for GA2 track 6000…
Predicting for GA3 track 0…
Predicting for GA3 track 1000…
Predicting for GA3 track 2000…
Predicting for GA3 track 3000…
Predicting for GA3 track 4000…
Predicting for GA3 track 5000…
Predicting for GA3 track 6000…
Predicting for H2O track 0…
Predicting for H2O track 1000…
Predicting for H2O track 2000…
Predicting for H2O track 3000…
Predicting for H2O track 4000…
Predicting for H2O track 5000…
Saved post-processed predictions → C:\Users\corna\honours\

In [30]:
# ── CELL 10  Assess ───────────────────────────────────────────────────────────
importlib.reload(tp)

for feature_set in fscs:
    curr_path = final_output_path / feature_set.get_id()
    
    for hyperparams in hps:
        curr_path = final_output_path / feature_set.get_id() / hyperparams.get_id()
        predictions_dir = curr_path / predictions_dir_ext
        probabilities_path = curr_path / probabilities_path_ext
        plot_path = curr_path / plot_path_ext
        report_path = curr_path / report_path_ext
        with open(report_path, "a") as f:
            f.write(f"Assessed at: {datetime.now()}\n")
            
        os.makedirs(predictions_dir, exist_ok=True)
        os.makedirs(curr_path, exist_ok=True)
        
        for pcon in ppcs: 
        
            tp.assess_performance(
                preds_dir  = predictions_dir,
                val_dir    = VAL_DIR,
                ppc        = pcon,
                prefixes   = TRAIN_PREFIXES,
                sources    = sources,
                report_path= report_path,
                plots      = plot_path,
                dwell_tags = ["wonderful"],
                rf_assess  = True,
                val_keys   = val_keys,    # ← restricts evaluation to val-only larvae
                train_keys = train_keys,  # ← raises error if any train larva leaked in,
    
)

c:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\tp_export.py:1183: DtypeWarning: Columns (0: behavior, 1: tags) have mixed types. Specify dtype option on import or set low_memory=False.
  results   = pd.read_csv(pred_path).copy()


[assess_performance] Val-key filter: 16859236 → 84865 rows (93 validation larvae)
[assess_performance] ✓ Contamination check passed — no train larvae in evaluation subset.


In [ ]:
# ── CELL 11  Plot grid ────────────────────────────────────────────────────────
importlib.reload(tp)

#tp.get_event_info(preds_dir=predictions_dir,
#                   val_dir    = VAL_DIR,
#                 ppc        = ppc,report_path= report_path,
#                 dwell_tags = ["wonderful"],val_keys   = val_keys,  
#                 train_keys = train_keys,)
# tp.plot_source_grid(predictions_dir, "GA1", predictions_dir, ppc, cols=10,val_keys=val_keys)
# tp.plot_source_grid(predictions_dir, "GA2", predictions_dir, ppc, cols=10,val_keys=val_keys)
tp.plot_source_grid(predictions_dir, "GA2", predictions_dir, ppc, cols=10,rows=250,val_keys=val_keys)

#tp.plot_source_grid(predictions_dir, "GA3", predictions_dir, ppc, cols=10,rows=100,val_keys=val_keys)

---
## Utilities

Run any of these stand-alone at any time.

In [ ]:
# Browse all cached feature columns
import feature_store as fs
df_cols = fs.list_cached_features(MASTER_CACHE_DIR)
print(f"{len(df_cols)} feature columns in cache")
df_cols

In [ ]:
# Health-check: are all ctx sources cached?
import feature_store as fs
fs.check_cache_health(MASTER_CACHE_DIR, ctx=ctx)

In [ ]:
# Load raw cached features for one source (for manual inspection / plotting)
import feature_store as fs
df_h2o = fs.load_source_features(MASTER_CACHE_DIR, "H2O")
print(df_h2o.shape)
df_h2o.head()

In [ ]:
# Load a saved FeatureSetConfig from a previous run
from feature_store import FeatureSetConfig
old_fsc = FeatureSetConfig.load(BASE_OUT_DIR / "RUN_xxx" / "feature_set_config.json")
print(old_fsc.summary())

---
## Optimization

In [ ]:
import importlib
import random
import os
import gc
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import feature_store as fs
importlib.reload(fs)
import tp_export as tp
from tp_export import PostProcessConfig
import split_dataset as sd
from pp import get_context, validate_context

TRAIN_DATA   = Path(r"C:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\annotations_split\train")
ANN_MASTER   = Path(r"C:\Users\corna\honours\fresh1\hp_2\data_intermediate\annotation\annotation.csv")
BASE_OUT_DIR = Path(r"C:\Users\corna\honours\fresh1\hp_2\data_intermediate\exported_models")
FOLDS_DIR= Path(r"C:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\annotations_split\three_folds")
MASTER_CACHE_DIR = BASE_OUT_DIR / "master_feature_cache"
LOGIC_FILE = "feature_registry"
FPS        = 6.0

f1 = {}
f2 = {}
f3 = {}

f1["pd"] = pd.read_csv(FOLDS_DIR / "fold_one" / "annotations_validation.csv")
f2["pd"] = pd.read_csv(FOLDS_DIR / "fold_two" / "annotations_train.csv")
f3["pd"] = pd.read_csv(FOLDS_DIR / "fold_three" / "annotations_validation.csv")

f1["train_ann"] = pd.merge(f2["pd"],f3["pd"],how="outer")
f2["train_ann"] =  pd.merge(f1["pd"],f3["pd"],how="outer")
f3["train_ann"] = pd.merge(f1["pd"],f2["pd"],how="outer")
f1["val_ann"]   = f1["pd"]
f2["val_ann"] = f2["pd"]
f3["val_ann"] = f3["pd"]
f1["VAL_DIR"] = Path(r"C:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\annotations_split\three_folds\fold_one")
f2["VAL_DIR"] = Path(r"C:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\annotations_split\three_folds\fold_two")
f3["VAL_DIR"] = Path(r"C:\Users\corna\honours\fresh1\hp_2\notebooks\parquet_pipeline\annotations_split\three_folds\fold_three")


for f in [f1,f2,f3]:
    f["TRAIN_PREFIXES"] = f["train_ann"]["source"].unique().tolist()
    f["TEST_FILES"]     = f["val_ann"]["source"].unique().tolist()
    f["sources"]        = list(set(f["TRAIN_PREFIXES"]).union(set(f["TEST_FILES"])))

    f["TRAIN_SLICES"] = {src: slice(None) for src in f["TRAIN_PREFIXES"]}
    f["train_keys"]   = f["train_ann"][["source", "ID"]].drop_duplicates()
    f["val_keys"]     = f["val_ann"][["source", "ID"]].drop_duplicates()


all_ann_keys = pd.merge(f1["train_keys"],f1["val_keys"],how="outer")

ctx = get_context(
    parquet_path = Path(r"C:\Users\corna\honours\fresh1\hp_2\data_intermediate\raw_trajectories.parquet"),
    ann_csv      = ANN_MASTER,
    session_dir  = TRAIN_DATA,
    sources      = f1["sources"],
)
clean_full_annotations = ctx.annotated.copy()

In [ ]:
import optuna
import os
import shutil
from pathlib import Path
importlib.reload(fs)
from feature_store import FeatureSetConfig
from tp_export import PostProcessConfig
import tp_export as tp
import tempfile
import statistics
from optuna.samplers import TPESampler

random_sampler = optuna.samplers.RandomSampler()

def objective(trial):
    # ==========================================
    # DOMAIN A: Feature Selection
    # ==========================================
    
    # N_OLD_TRIALS = 550
    # def penalize_early_trials(n_total_trials: int) -> np.ndarray:
    #     weights = np.ones(n_total_trials)
        
    #     weights[:N_OLD_TRIALS] = 0.2
        
    #     return weights

    # if trial.number % 15 > 10:
    #     trial.study.sampler = random_sampler
    # else:
    #     trial.study.sampler = TPESampler(weights=penalize_early_trials)
    
    standard_wins = [5,10,13,17,23,30,43,50,65,70,80,95,110,120]#[5,7,10,11,13,15,17,18,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105,110,115,120]
    heavy_wins = [w for w in standard_wins if w <= 50]
    
    available_features = {
        "omega_body_mean": standard_wins,
        "omega_head_std": standard_wins,
        "omega_relative_mean": standard_wins,
        "rog": standard_wins,
        "tortuosity": standard_wins,
        "msd": standard_wins,
        "bending_std": standard_wins,
        "hc_ratio_mean": standard_wins,
        "ht_ratio_mean": standard_wins,
        "omega_body_mean_slope_smooth": standard_wins,
        "vel_mean": standard_wins,
        "vel_std": standard_wins,
        "vel_norm_mean": standard_wins,
        "head_vel_mean": standard_wins,
        "head_vel_std": standard_wins,
        "omega_heading_mean": standard_wins,
        "coverage": standard_wins,
        "bend_peaks_rate": standard_wins,
        "pause_run_frac": standard_wins,
        "reversal_rate": standard_wins,
        "angular_tortuosity": standard_wins,
        "revisitation_mean": heavy_wins,
        "revis_slope_smooth": standard_wins,
        "rog_slope_smooth": standard_wins,
        "tort_slope_smooth": standard_wins,
        "path_efficiency": heavy_wins,
        "vel_autocorr": standard_wins,
        "pca_ratio": heavy_wins,
        "pca_area_norm": heavy_wins,
        "net_displacement_norm": standard_wins,
        "turn_fraction": standard_wins,
        "posture_asymmetry_max": standard_wins,
        "pause_fraction": standard_wins,
        "omega_heading_max": standard_wins,
        "curvature_index_mean": standard_wins,
        "head_bend_max": standard_wins,
        "path_length_norm": standard_wins,
        "turn_event_alternation": heavy_wins,
        "turn_event_interval_cv": heavy_wins,
        "turn_event_amp_mean": heavy_wins,
        "turn_event_gap_max_bl": heavy_wins,
        "turn_event_gap_mean_bl": heavy_wins,
        "turn_event_count": heavy_wins,
        # Restricted features
        "vel_lag": [50,55,60,65,70,75,80,85,90,95,100,105,110,115,120],
        "vel_lead": [50,55,60,65,70,75,80,85,90,95,100,105,110,115,120],
        "bend_freq_rolling": [60,65,70,75,80,85,90,95,100,105,110,115,120]
    }
    dynamic_windowed_features = {}
    MAX_WINDOWS_PER_FEATURE = 3

    for feat_name, valid_wins in available_features.items():
        
        max_possible = min(MAX_WINDOWS_PER_FEATURE, len(valid_wins))
        n_windows = trial.suggest_int(f"n_wins_{feat_name}", 0, max_possible)

        if n_windows == 0:
            continue 

        selected_wins = []
        current_min_idx = 0
        
        for i in range(n_windows):
            max_idx = len(valid_wins) - 1 - (n_windows - 1 - i)
        
            win_idx = trial.suggest_int(f"idx_{feat_name}_w{i}", current_min_idx, max_idx)
            selected_wins.append(valid_wins[win_idx])
            
            current_min_idx = win_idx + 1 

        dynamic_windowed_features[feat_name] = selected_wins

    dynamic_base_features = []
    if trial.suggest_categorical("use_larva_body_length", [True, False]):
        dynamic_base_features.append("larva_body_length")

    if not dynamic_base_features and not dynamic_windowed_features:
        raise optuna.exceptions.TrialPruned()

    fsc = FeatureSetConfig(
        name=f"optuna_trial_{trial.number}",
        description="Granular Per-Feature-Window Search",
        base_features=dynamic_base_features,
        windowed_features=dynamic_windowed_features
    )
    # ==========================================
    # DOMAIN B: Tree Hyperparameters
    # ==========================================
        
    min_leaf = trial.suggest_float("min_leaf", 0.0005, 0.05)
    split_multiplier = trial.suggest_float("split_multiplier", 2.0, 10.0)

    rf_params = {
        "n_estimators": 150,# trial.suggest_int("n_estimators", 50, 300, step=50),
        "max_depth": 35, #trial.suggest_int("max_depth", 5, 45),
        "min_leaf": min_leaf,
        "min_split": min_leaf * split_multiplier,}
    
    rf_params["max_feat"] = trial.suggest_categorical(
        "max_feat", 
        ['sqrt', 'log2', 0.2, 0.3, 0.4, 0.5]
)
    # ==========================================
    # DOMAIN C: Post-Processing Parameters
    # ==========================================
    MF = 15     #trial.suggest_int("MF", 1, 17, step=2)      # Median Filter (odd numbers)
    TH = 0.61   # trial.suggest_float("TH", 0.4, 0.8)        # Threshold
    GC = 4 #trial.suggest_int("GC", 1, 15,step=2)              # Gap Close
    GO = 14 #trial.suggest_int("GO", 1, 25,step=2)             # Gap Open
    
    ppc = PostProcessConfig(MF, TH, GC, GO)

    # ==========================================
    # EXECUTION: Train -> Infer -> Predict -> Assess
    # ==========================================
    # Create an isolated output directory so trials don't overwrite each other
    for fold in [f1,f2,f3]:
    
        TRAIN_SLICES = fold["TRAIN_SLICES"]
        TRAIN_PREFIXES = fold["TRAIN_PREFIXES"]
        TEST_FILES = fold["TEST_FILES"]
        train_keys = fold["train_keys"]
        val_keys = fold["val_keys"]
        VAL_DIR = fold["VAL_DIR"]
        sources        = list(set(TRAIN_PREFIXES).union(set(TEST_FILES)))
        
        
        with tempfile.TemporaryDirectory() as temp_dir:
            temp_out_dir = Path(temp_dir)
            
            model_path = temp_out_dir / "final_model.pkl"
            probabilities_path = temp_out_dir / "probabilities.csv"
            preds_dir = temp_out_dir / "preds"
            os.makedirs(preds_dir, exist_ok=True)
            
            prefixes = list(TRAIN_SLICES.keys())
            trial_seed = globals().get('SEED', globals().get('seed', 42))
            
            # 1. Train
            tp.train(
                ctx=ctx, slices=TRAIN_SLICES, prefixes=prefixes, logic_file=LOGIC_FILE,
                model_path=model_path, feature_path=temp_out_dir / "fsc.pkl",
                metadata_path=temp_out_dir / "meta.pkl", seed=trial_seed,
                cache_dir=MASTER_CACHE_DIR, fsc=fsc, train_keys=train_keys, 
                do_plot_gini=False, plot_path=temp_out_dir,
                n_est=rf_params['n_estimators'], max_dep=rf_params['max_depth'],
                min_split=rf_params['min_split'], min_samples=rf_params['min_leaf'],
                max_feat=rf_params['max_feat'],
            )
            
            # 2. Infer
            tp.infer(
                model_path=model_path, ctx=ctx, files=TEST_FILES,
                feature_path=temp_out_dir / "fsc.pkl", probabilities_path=probabilities_path,
                metadata_test_path=temp_out_dir / "meta_test.pkl",
                cache_dir=MASTER_CACHE_DIR, logic_file=LOGIC_FILE, fsc=fsc,
                infer_keys=val_keys
            )
            
            # 3. Predict (Post-process)
            tp.predict(
                probabilities_path=probabilities_path, ppc=ppc,
                predictions_dir=preds_dir, ctx=ctx,
                logic_file=LOGIC_FILE, plot=False
            )
            
            # 4. Assess
            fold["metrics"] = tp.assess_performance(
                preds_dir=preds_dir, val_dir=VAL_DIR,
                ppc=ppc, prefixes=TRAIN_PREFIXES, sources=sources,
                report_path=temp_out_dir / "log.txt", plots=None,
                dwell_tags=["wonderful"], rf_assess=True,
                val_keys=val_keys, train_keys=train_keys
            )
            
            
    
    # Store auxiliary metrics in Optuna dashboard
    for metric_name, value in f1["metrics"].items():
        if isinstance(value, (int, float)): 
            val = statistics.mean([value,f2["metrics"][metric_name],f3["metrics"][metric_name]])
            trial.set_user_attr(metric_name, val)
            
    
    return statistics.mean([f1["metrics"]["f1"], f2["metrics"]["f1"], f3["metrics"]["f1"]])

In [ ]:
import optuna
from optuna.samplers import TPESampler
import warnings

# 1. Name your study and create a local database file
study_name = "dwelling_mini_search_v31"
storage_name = f"sqlite:///{study_name}.db?timeout=60"


warnings.filterwarnings("ignore", category=FutureWarning, module="optuna")
N_OLD_TRIALS = 550

def penalize_early_trials(n_total_trials: int) -> np.ndarray:
    weights = np.ones(n_total_trials)
    
    weights[:N_OLD_TRIALS] = 0.2
    
    return weights

random_sampler = optuna.samplers.RandomSampler()

study = optuna.create_study(
    study_name=study_name,
    storage=storage_name,
    direction="maximize",
    load_if_exists=True,
    sampler=TPESampler(n_startup_trials=182),#weights=penalize_early_trials)
    #sampler = random_sampler
)
study.set_user_attr("name",study_name)
study_dir  = Path(BASE_OUT_DIR / "optuna" / study_name)
study_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
importlib.reload(tp)
import os
import warnings
warnings.filterwarnings(
    "ignore", 
    message="`sklearn.utils.parallel.delayed` should be used with"
)
os.environ["PYTHONWARNINGS"] = (
    "ignore::UserWarning:sklearn.utils.parallel"
)

ann = ctx.annotated[["source", "ID", "et", "behavior", "tags"]].copy()
ann["source"] = ann["source"].astype(str)
ann["ID"]     = ann["ID"].astype(str)
ann["et"]     = ann["et"].astype("float32").round(4)

if all_ann_keys is not None:
    print("[train] Pre-filtering annotation table to annotated train larvae …")
    ann_keys = all_ann_keys.copy()
    ann_keys["source"] = ann_keys["source"].astype(str)
    ann_keys["ID"]     = ann_keys["ID"].astype(str)
    n_ann_before = len(ann)
    ann = ann.merge(ann_keys, on=["source", "ID"], how="inner")
    print(f"[train] Annotations restricted: {n_ann_before} → {len(ann)} rows "
            f"({ann[['source','ID']].drop_duplicates().shape[0]} larvae)")

ctx.annotated = ann
    
# 3. Run the optimization.
# Instead of n_trials, you can use 'timeout' (in seconds). 
# For example, let it run for exactly 12 hours (60 * 60 * 12)
print(f"Starting Mega-Search. Current best F1: {study.best_value if len(study.trials) > 0 else 'None'}")

try:
    study.optimize(objective, timeout=43200)#n_trials=1)#timeout=43200) # 12 hours
except KeyboardInterrupt:
    print("\nSearch paused safely by user.")

print("\n=== ABSOLUTE OPTIMUM FOUND ===")
print(f"Best F1 Score: {study.best_value}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")